# Intro

Recoding of https://github.com/Daniel-EST/deep-steganography/ but in pytorch while adding below to fit paper being audited as code was not available to test

from paper: 500 training images, 50 validation images, and 50 test images, all uniformly resized to a manageable dimension of 64x64x3 pixels

text > Huffman > LSB > NN Embed > NN decode > LSB > Huffman > text

LSB - use pillow library (https://pypi.org/project/pillow/)

Huffman - use geeks for geeks

In [42]:
from LSBSteg.LSBSteg import LSBSteg
from dahuffman import HuffmanCodec

# code example from https://github.com/RobinDavid/LSB-Steganography libraru, lsb is using
# LSB 
def LSBEmbed(text,image):
    # Embed text into image using least sig bit method
    steg = LSBSteg(image)
    img_encoded = steg.encode_binary(text)
    return img_encoded

def LSBExtract(image):
    # pull least sig bit from image and reformat into embedded text
    steg = LSBSteg(image)
    extracted_text = steg.decode_binary()
    return extracted_text

# huffman serialization assisted with chatgpt for debugging, dahuffman use from PyPl example
def HuffmanEncode(data):
    # encode text into huffman tree + map
    codec = HuffmanCodec.from_data(data)
    encoded_data = codec.encode(data)

    # finally prepend all so tree:data is one block
    return codec, encoded_data

def HuffmanDecode(codec,data):
    # using Huffman map reconstruct text to original values and decode / unzip data
    decoded_data = codec.decode(data)
    return decoded_data

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import os, certifi
import numpy as np
import cv2
import time
import random

os.environ['SSL_CERT_FILE'] = certifi.where()
# option 1 uses "Hello World!" for all to reuse codec and make things simpler
tiny_imageNet = load_dataset("zh-plus/tiny-imagenet")
num_images = 10

total_images = len(tiny_imageNet['train'])
# Shuffle indices and split
all_indices = list(range(total_images))
random.seed( time.process_time())
random.shuffle(all_indices)
cover_indices = all_indices[:num_images]
secret_indices = all_indices[num_images:num_images + num_images]

# Sampled images
cover_images =  [tiny_imageNet['train'][i]['image'] for i in cover_indices]
secret_images = [tiny_imageNet['train'][i]['image'] for i in secret_indices]

ex_data = "Hello World!"
codec,encoded_data = HuffmanEncode(ex_data)

# count len(tiny_imageNet['train'])
# fig,axes = plt.subplots(1, 2,figsize=(4, 2))
steno_images = []
for i in range(0,num_images):
    image = tiny_imageNet['train'][i]['image']
    np_image = np.array(image)
    steg_img = LSBEmbed(encoded_data,np_image)
    steno_images[i] = steg_img



In [ ]:
# steg NN